In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workingspace.bronze.raw_files;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workingspace.bronze.orders_copy (
    order_id      BIGINT,
    customer_id   BIGINT,
    order_status  STRING,
    order_amount  DECIMAL(10,2),
    country       STRING,
    order_date    DATE
)
USING DELTA;

In [0]:
csv_content_1 = """order_id,customer_id,order_status,order_amount,country,order_date
101,201,PLACED,450.00,IN,2026-07-25
102,202,SHIPPED,220.50,US,2026-07-25
103,203,PLACED,175.00,UK,2026-07-25"""

dbutils.fs.put("/Volumes/workingspace/bronze/raw_files/batch1.csv", csv_content_1, overwrite=True)

In [0]:
%sql
-- COPY INTO workingspace.bronze.orders_copy
-- FROM '/Volumes/workingspace/bronze/raw_files/'
-- FILEFORMAT = CSV
-- FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
-- COPY_OPTIONS ('mergeSchema' = 'true');

COPY INTO workingspace.bronze.orders_copy
FROM (
  SELECT 
    CAST(order_id AS BIGINT) AS order_id,
    CAST(customer_id AS BIGINT) AS customer_id,
    order_status,
    CAST(order_amount AS DECIMAL(10,2)) AS order_amount,
    country,
    CAST(order_date AS DATE) AS order_date
  FROM '/Volumes/workingspace/bronze/raw_files/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('force'='true');


In [0]:
%sql
select * from workingspace.bronze.orders_copy;

In [0]:
csv_content_2 = """order_id,customer_id,order_status,order_amount,country,order_date
104,204,PLACED,300.00,IN,2026-07-27
105,205,DELIVERED,90.00,US,2026-07-27"""

dbutils.fs.put("/Volumes/workingspace/bronze/raw_files/batch2.csv", csv_content_2, overwrite=True)

In [0]:
%sql
SELECT * FROM workingspace.bronze.orders_copy ORDER BY order_id;

In [0]:
%sql
DESCRIBE EXTENDED workingspace.bronze.orders_copy

In [0]:
%sql
select count(*) from workingspace.bronze.orders_copy;

In [0]:
%sql
DROP TABLE IF EXISTS workingspace.bronze.orders_staging;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workingspace.bronze.orders_staging (
    order_id      BIGINT,
    customer_id   BIGINT,
    order_status  STRING,
    order_amount  DECIMAL(10,2),
    country       STRING,
    order_date    DATE,
    create_at    TIMESTAMP
);

INSERT INTO workingspace.bronze.orders_staging VALUES
(2, 102, 'DELIVERED', 540.50, 'US', '2026-07-21', current_timestamp()),  -- existing order, status changed
(5, 105, 'PLACED', 610.00, 'IN', '2026-07-27', current_timestamp());     -- brand new order

In [0]:
%sql
select * from workingspace.bronze.orders;

In [0]:
%sql
select * from workingspace.bronze.orders_staging;

In [0]:
%sql
MERGE INTO workingspace.bronze.orders as sources
USING workingspace.bronze.orders_staging as destination
on sources.order_id = destination.order_id
WHEN MATCHED THEN
     UPDATE SET 
         sources.order_status = destination.order_status,
         sources.order_amount = destination.order_amount,
         sources.create_at = destination.create_at
WHEN NOT MATCHED THEN
    INSERT (order_id,customer_id,order_status,order_amount,country,order_date,create_at)
    values (destination.order_id,destination.customer_id,destination.order_status,destination.order_amount,destination.country,destination.order_date,destination.create_at)

In [0]:
%sql
select * from workingspace.bronze.orders;

In [0]:
%sql
desc extended workingspace.bronze.orders;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workingspace.bronze.orders_optimize_test (
    order_id      BIGINT,
    customer_id   BIGINT,
    order_status  STRING,
    order_amount  DECIMAL(10,2),
    country       STRING,
    order_date    DATE
)
USING DELTA;

In [0]:
%sql
INSERT INTO workingspace.bronze.orders_optimize_test VALUES (201, 301, 'PLACED', 100.00, 'IN', '2026-07-01');
INSERT INTO workingspace.bronze.orders_optimize_test VALUES (202, 302, 'PLACED', 150.00, 'US', '2026-07-02');
INSERT INTO workingspace.bronze.orders_optimize_test VALUES (203, 303, 'PLACED', 200.00, 'UK', '2026-07-03');
INSERT INTO workingspace.bronze.orders_optimize_test VALUES (204, 304, 'PLACED', 250.00, 'IN', '2026-07-04');
INSERT INTO workingspace.bronze.orders_optimize_test VALUES (205, 305, 'PLACED', 300.00, 'US', '2026-07-05');

In [0]:
%sql
DESC extended workingspace.bronze.orders_optimize_test;

In [0]:
%sql
DESCRIBE DETAIL workingspace.bronze.orders_optimize_test;

In [0]:
%sql
OPTIMIZE workingspace.bronze.orders_optimize_test;

In [0]:
%sql
DESCRIBE DETAIL workingspace.bronze.orders_optimize_test;

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM workingspace.bronze.orders_optimize_test RETAIN 0 HOURS;

In [0]:
%sql
DESC HISTORY workingspace.bronze.orders_optimize_test;

In [0]:
%sql
select * from workingspace.bronze.orders_optimize_test version as of 9;